In [ ]:
import os
from typing import List, Optional
from dotenv import load_dotenv
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.graphs import Neo4jGraph

load_dotenv()

NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USER = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE")

graph = Neo4jGraph(url=NEO4J_URI, username=NEO4J_USER, password=NEO4J_PASSWORD, database=NEO4J_DATABASE)
llm = ChatOllama(model="mistral", temperature=0)

### Overall Request Payload Format

```json
{
  "model": "...",
  "messages": [
    { "role": "system", "content": "You extract knowledge graphs." },
    { "role": "user", "content": "Apple manufactures the iPhone" }
  ],
  "response_format": "...",
  "tools": [
    {
      "type": "function",
      "function": {
        "name": "create_edge",
        "parameters": { "...": "json schema here" }
      }
    }
  ],
  "temperature": 0,
  "max_output_tokens": 500
}
```

### Response Format

```json
"response_format": {
  "type": "json_schema",
  "json_schema": {
    "name": "KnowledgeGraph",
    "schema": {

      "$defs": {

        "Entity": {
          "type": "object",
          "properties": {
            "name": {
              "type": "string",
              "description": "The name of the entity. For example: 'Apple Inc'"
            },
            "type": {
              "type": "string",
              "description": "The category. For example: 'Organization', 'Person', 'Product'"
            }
          },
          "required": ["name", "type"]
        },

        "Relationship": {
          "type": "object",
          "properties": {
            "source": { "$ref": "#/$defs/Entity" },
            "target": { "$ref": "#/$defs/Entity" },
            "relation": {
              "type": "string",
              "description": "The verb connecting them. For example: 'manufactures', 'works_for'"
            }
          },
          "required": ["source", "target", "relation"]
        }
      },

      "type": "object",
      "properties": {
        "entities": {
          "type": "array",
          "items": { "$ref": "#/$defs/Entity" }
        },
        "relationships": {
          "type": "array",
          "items": { "$ref": "#/$defs/Relationship" }
        }
      },

      "required": ["entities", "relationships"]
    }
  }
}

### Tools Format

```json
"tools": [
  {
    "type": "function",
    "function": {
      "name": "create_edge",
      "description": "Create a relationship between two entities in the knowledge graph",
      "parameters": {
        "type": "object",
        "properties": {
          "source": {
            "type": "object",
            "description": "The source entity",
            "properties": {
              "name": {
                "type": "string",
                "description": "Canonical entity name, e.g. 'Apple Inc'"
              },
              "type": {
                "type": "string",
                "description": "Entity category such as Organization, Person, Product"
              }
            },
            "required": ["name", "type"]
          },
          "target": {
            "type": "object",
            "description": "The target entity",
            "properties": {
              "name": {
                "type": "string",
                "description": "Canonical entity name, e.g. 'iPhone'"
              },
              "type": {
                "type": "string",
                "description": "Entity category such as Product"
              }
            },
            "required": ["name", "type"]
          },
          "relation": {
            "type": "string",
            "description": "Specific verb describing the relationship, e.g. 'manufactures', 'acquired', 'founded'"
          }
        },
        "required": ["source", "target", "relation"]
      }
    }
  }
]

In [ ]:
# We don't want model to write sentences, but a graph-shaped object

from pydantic import BaseModel, Field

class Entity(BaseModel):
    name: str = Field(description="The name of the entity. For example: 'Apple Inc'") # The description is an instruction for llm
    type: str = Field(description="The category. For example: 'Organization', 'Person', 'Product'")

class Relationship(BaseModel):
    source: Entity
    target: Entity
    relation: str = Field(description="The verb connecting them. For example: 'manufactures', 'works_for'")

class KnowledgeGraph(BaseModel):
    entities: List[Entity]
    relationships: List[Relationship]

Example of an json output to be converted to a KG object:
```json
{
  "entities": [
    { "name": "Apple Inc", "type": "Organization" },
    { "name": "iPhone 15", "type": "Product" },
    { "name": "Steve Jobs", "type": "Person" },
    { "name": "Steve Wozniak", "type": "Person" },
    { "name": "Cupertino", "type": "Location" },
    { "name": "California", "type": "Location" },
    { "name": "2023", "type": "Date" }
  ],
  "relationships": [
    {
      "source": { "name": "Apple Inc", "type": "Organization" },
      "target": { "name": "iPhone 15", "type": "Product" },
      "relation": "released"
    },
    {
      "source": { "name": "Apple Inc", "type": "Organization" },
      "target": { "name": "Steve Jobs", "type": "Person" },
      "relation": "founded_by"
    },
    {
      "source": { "name": "Apple Inc", "type": "Organization" },
      "target": { "name": "Steve Wozniak", "type": "Person" },
      "relation": "founded_by"
    },
    {
      "source": { "name": "Apple Inc", "type": "Organization" },
      "target": { "name": "Cupertino", "type": "Location" },
      "relation": "headquartered_in"
    },
    {
      "source": { "name": "Cupertino", "type": "Location" },
      "target": { "name": "California", "type": "Location" },
      "relation": "located_in"
    },
    {
      "source": { "name": "iPhone 15", "type": "Product" },
      "target": { "name": "2023", "type": "Date" },
      "relation": "released_in"
    }
  ]
}
```

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# Turns into an object generator
structured_llm = llm.with_structured_output(KnowledgeGraph)

def extract_entities_and_relations(text_chunk: str) -> KnowledgeGraph:
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are an experienced and detail-oriented Knowledge Graph engineer. Extract ONLY the most important entities and their direct relationships from the text."),
        ("human", "Text to process: {input}")
    ])
    
    chain = prompt | structured_llm
    kg_object = chain.invoke({"input": text_chunk})

    print("\nLLM GENERATED JSON:")
    print(kg_object.model_dump_json(indent=2)) 
    
    return kg_object

In [ ]:
# Converts plain KG into Neo4j db nodes and edges

def store_in_neo4j(kg: KnowledgeGraph):
    for relationship in kg.relationships:
        rel_type = relationship.relation.replace(' ', '_').upper()

        standard_cypher = f"""
        // MERGE means match or create this exact pattern
        // s is just a variable name for the node inside this query
        // Create node, attach Entity label whose name equals $source_name. Arbitrary
        MERGE (s:Entity {{name: $source_name}}) SET s.type = $source_type
        MERGE (t:Entity {{name: $target_name}}) SET t.type = $target_type
        MERGE (s)-[r:`{rel_type}`]->(t)
        """
        graph.query(standard_cypher, params={
            "source_name": relationship.source.name.strip().title(),
            "source_type": relationship.source.type.upper(),
            "target_name": relationship.target.name.strip().title(),
            "target_type": relationship.target.type.upper()
        })

In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

def run_document_pipeline(pdf_path: str):
    loader = PyMuPDFLoader(pdf_path)
    docs = loader.load() # List of pages
    splitter = RecursiveCharacterTextSplitter(chunk_size=1200, chunk_overlap=150)
    chunks = splitter.split_documents(docs)
        
    for i, chunk in enumerate(chunks):
        print(f"Working on chunk {i+1}...")
        graph_data = extract_entities_and_relations(chunk.page_content)
        store_in_neo4j(graph_data)
        
run_document_pipeline("data/document.pdf")

In [ ]:
from langchain_neo4j import Neo4jGraph, GraphCypherQAChain
from langchain_core.prompts import ChatPromptTemplate

CYPHER_SYSTEM_TEMPLATE = (
"""Task: Generate a Cypher statement to query a graph database.
Instructions:
Use only the provided relationship types and properties in the schema.
Do not use any other relationship types or properties that are not provided.

Schema:
{schema}

Important Rules:
1. In this database, all nodes are labeled as 'Entity'. 
2. The specific category is stored in the 'type' property (e.g., n.type = 'Methodology'). 
3. Do NOT use specific node labels like :Person or :Methodology. Use (e:Entity {{type: 'Methodology'}}).
4. Return only the Cypher query, no explanations."""
)

def ask_the_graph(question: str):
    cypher_prompt = ChatPromptTemplate.from_messages([
        ("system", CYPHER_SYSTEM_TEMPLATE),
        ("human", "{question}")
    ])
    
    graph.refresh_schema()
    
    chain = GraphCypherQAChain.from_llm(
        llm=llm, 
        graph=graph, 
        verbose=True, 
        cypher_prompt=cypher_prompt,
        allow_dangerous_requests=True
    )
    response = chain.invoke({"query": question})
    return response["result"]

ask_the_graph("What are the primary neural network architectures used?")

In [ ]:
from pyvis.network import Network

def visualize_results():
    query = "MATCH (n)-[r]->(m) RETURN n.name as source, type(r) as rel, m.name as target LIMIT 100"
    results = graph.query(query)
    net = Network(notebook=True, cdn_resources="remote", directed=True, bgcolor="#ffffff", font_color="black", height="600px")
    
    for res in results:
        net.add_node(res['source'], label=res['source'], color="#4287f5")
        net.add_node(res['target'], label=res['target'], color="#f54242")
        net.add_edge(res['source'], res['target'], label=res['rel'])
    
    return net.show("knowledge_graph.html")

visualize_results()